# LLaMA!!!! - 2023

So the LLaMA architecture is simply the same as the previous models like GPT3,.. and so on, but it is just tweaked and modified on some impactful and fruitful parts including: 
- Use the Pre-Norm instead of Post-norm for training stability and good for warmup scheduling.
- Use the SwiGlU as an Activation function instead of the conventional GELU or ReLU, good for budget, and cranked down the FFN multiplication from 4x to just 2/3x 
- RoPE (RoFormer's Positional Embedding)

In [1]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
from pydantic import BaseModel, Field, model_validator
import math

In [12]:
class LLaMAConfig(BaseModel):
    vocab_size: int = Field(default = 32000, ge = 0, description = "Vocabulary Size")
    context_length: int = Field(default = 2048, gt= 0, description = "Max content length or block size")
    d_model: int = Field(default = 4096, gt = 0, description = "Model/hidden dimension (hidden_size)")
    n_layer: int = Field(default = 32, gt = 0, description = "Number of stacked decoder within the model")
    num_head: int = Field(default = 32, gt = 0, description = "Number of attention heads")
    intermediate_step: int = Field(default = 11008, gt = 0, description = "FFN Inner dimension")
    dropout: float = Field(default = 0.0, ge = 0.0, description = "Residual Dropout")
    attention_dropout: float = Field(default = 0.0, ge = 0.0, description = "Embedding Dropout")
    rms_norm_eps: float = Field(default = 1e-6, ge = 0.0, description = "eps for the RMS")


model_config = {"frozen": True}

@model_validator(mode = "after")

def check_head_divide_embedding(self):
    if self.d_model % self.num_head != 0: 
        raise ValueError(
            f"d_model ({self.d_model}) must be divisible by num_head ({self.num_head});"
            f"got remainder {self.d_model % self.num_head}"
        )
    return self




### RoPE Function

In [ ]:
class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len, theta = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0 # the head_dim must be even for the RoPE

        freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))

        # angle for every (position, freq) pairs:
        positions = torch.arange(max_seq_len).float()
        angles = torch.outer(positions, freqs)

        # cache as buffers so they move with .to(device) and are not trained
        self.register_buffer("cos", torch.cos(angles), persistent = False)
        self.register_buffer("sin", torch.sin(angles), persistent = False)

    def forward(self, T):
        return self.cos[:T], self.sin[:T]

def apply_rope(q, k, rope_cos, rope_sin):
    cos = rope_cos.unsqueeze(0).unsqueeze(0)
    sin = rope_sin.unsqueeze(0).unsqueeze(0) # [1, 1, T, head_dim/2]

    def rotate(x):
        x1 = x[..., 0::2] # even dim
        x2 = x[..., 1::2] # odd dim 

        rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim = -1)

        return rotated.flatten(-2) # flatten back to [B, H, T, head_dim]

    return rotate(q), rotate(k)



### Attention

In [ ]:
class MaskedMultiSelfAttention(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        assert config.d_model % config.num_head == 0
        self.num_head = config.num_head
        self.d_model = config.d_model
        self.head_dim = config.d_model // config.num_head

        self.qkv_proj = nn.Linear(config.d_model, 3 * config.d_model, bias = False)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias = False)

        self.attention_dropout = nn.Dropout(config.attention_dropout)
        self.residual_dropout = nn.Dropout(config.residual_dropout)
        self.out_proj.RESIDUAL_SCALE_INIT = True

        self.rope = RoPE(self.head_dim, config.context_length)

        causal_mask = torch.tril(torch.ones(config.context_length, config.context_length))
        self.register_buffer(
            "causal_mask", causal_mask.view(1, 1, config.context_length, config.context_lenght)
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(C, dim = 2)

        q = q.view(B, T, self.num_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_head, self.head_dim).transpose(1, 2)

        rope_cos, rope_sin = self.rope(T)
        
        q, k = apply_rope(q, k, rope_cos, rope_sin)
        
        attention = q @ k.tranpose(-2, -1) / math.sqrt(self.head_dim) # [B, H, T, D] @ [B, H, D, T] -> [B, H, T, T]
        attention = attention.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attention = self.attention_dropout(F.softmax(attention, dim = -1))

        score = attention @ v # [B, H, T, T] @ [B, H, T, D] -> [B, H, T, D]

        score = score.transpose(1, 2).contiguous().view(B, T, C)

        return self.residual_dropout(self.out_proj(score))

In [ ]:
class PositionWiseFNN(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()

        hidden_dim = int(2 * config.intermediate_step / 3)

        multiple_of = getattr(config, "multiple_of", 256)

        hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of) 

        
        self.fc1 = nn.Linear(config.hidden_size, hidden_dim, bias = False)
        self.fc2 = nn.Linear(hidden_dim, config.hidden_size, bias = False)
        self.fc3 = nn.Linear(config.hidden_size, hidden_dim, bias = False)

        self.dropout = nn.Dropout(config.dropout)
        self.fc2.RESIDUAL_SCALE_INIT = True

    def forward(self, x):
        gate = F.silu(self.fc1(x))
        value = self.fc3(x)
        x = self.fc2(gate * value)
        return self.dropout(x)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # RMS statistic: no mean-centering, unlike LayerNorm
        norm = x * torch.rsqrt(x.pow(2).mean(dim = -1,  keepdim = True) + self.eps)
        return norm * self.weight

    
class TransformerBlock(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.attention_norm = RMSNorm(config.hidden_size, eps = config.rms_norm_eps)
        self.attention = MaskedMultiSelfAttention(config)

        self.ffn_norm = RMSNorm(config.hidden_size, eps = config.rms_norm_eps)
        self.feed_forward = PositionWiseFNN(config)

    def forward(self, x):
        x = x + self.attention(self.attention_norm(x))
        x = x + self.feed_forward(self.ffn_norm)
        return x 
    